# Project tips

A few tips based on reviewing your Milestone 2.

## Data access

As a reminder, if you access files by mounting your google drive, I **cannot** access them to run your code!

Instead, you can create a public link to the file and load from your code, as in the example below:


In [17]:
import pandas as pd

# Google Drive direct download links have a specific format.
# A typical shareable link looks like: https://drive.google.com/file/d/FILE_ID/view?usp=sharing
# To get a direct download link, you need to extract the FILE_ID and use this format:
# https://drive.google.com/uc?export=download&id=FILE_ID

# Replace 'YOUR_FILE_ID_HERE' with the actual file ID from your Google Drive shareable link.
# For example, if your shareable link is 'https://drive.google.com/file/d/12345abcdeFGHIJKLMNOPQRSTUVW/view?usp=sharing',
# then YOUR_FILE_ID_HERE would be '12345abcdeFGHIJKLMNOPQRSTUVW'.
# e.g. https://drive.google.com/file/d/1o0DE3VtvlN3IpAxkUykzjoXDtuzu8L8X/view?usp=sharing
file_id = '1o0DE3VtvlN3IpAxkUykzjoXDtuzu8L8X' # <--- Replace this with your actual file ID

download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

try:
    # Read the CSV file directly into a pandas DataFrame
    df = pd.read_csv(download_url)
    print("CSV file successfully loaded into DataFrame.")
    display(df.head())
except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the file ID is correct and the file is publicly accessible (or shared with 'Anyone with the link').")
    print("Also, verify that the link points to a CSV file.")

CSV file successfully loaded into DataFrame.


,col1,col2,col3
0,v1,v2,v3
1,v4,v5,v6


If you have many files and/or large files, you can store them as a zip file in your drive, create a public link, then unzip to the workspace, like the example below

In [18]:
import requests
import zipfile
import os

# --- IMPORTANT ---
# Replace 'YOUR_FILE_ID_HERE' with the actual file ID from your Google Drive shareable link.
# Ensure the file is publicly accessible or shared with 'Anyone with the link'.
# e.g., https://drive.google.com/file/d/1YrwPpOhzMJwaCgEgagQT0xtVCFnZxNsx/view?usp=sharing

file_id = '1YrwPpOhzMJwaCgEgagQT0xtVCFnZxNsx' # <--- Replace this with your actual file ID for the zip file

download_url = f'https://drive.google.com/uc?export=download&id={file_id}'

zip_file_name = 'downloaded_archive.zip'
extraction_path = './extracted_data'

try:
    # Download the zip file
    print(f"Downloading zip file from: {download_url}")
    response = requests.get(download_url, stream=True)
    response.raise_for_status() # Raise an exception for bad status codes

    with open(zip_file_name, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"'{zip_file_name}' downloaded successfully.")

    # Unzip the file
    print(f"Unzipping '{zip_file_name}' to '{extraction_path}'...")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print("File unzipped successfully.")

    # List contents of the extracted directory (optional)
    print(f"Contents of '{extraction_path}':")
    for root, dirs, files in os.walk(extraction_path):
        for name in files:
            print(os.path.join(root, name))
        for name in dirs:
            print(os.path.join(root, name))

except requests.exceptions.RequestException as e:
    print(f"Error downloading file: {e}")
    print("Please check the file_id and ensure the file is publicly accessible.")
except zipfile.BadZipFile:
    print(f"Error: '{zip_file_name}' is not a valid zip file.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


'downloaded_archive.zip' downloaded successfully.
Unzipping 'downloaded_archive.zip' to './extracted_data'...
File unzipped successfully.
Contents of './extracted_data':
./extracted_data/data
./extracted_data/data/test3.csv
./extracted_data/data/test.csv
./extracted_data/data/test2.csv
./extracted_data/data/test4.csv


In [15]:
!ls extracted_data/data

test2.csv  test3.csv  test4.csv  test.csv


In [14]:
# then you can read a file locally
pd.read_csv('extracted_data/data/test2.csv')

,col1,col2,col3
0,v1,v2,v3
1,v4,v5,v6


## Tools for structured data

For structured data, RAG may not be the right solution.

E.g., if you want to answer questions like "How many times does X happen in the data?" or "Find all songs with property Y", RAG will just return a small set of rows that match your query.

Eg for data like

| artist | song | genre |
|--------|------|-------|
| bob marley | i shot the sheriff | reggae |
| allen toussaint | soul sister | soul |
...


if you do a vector search for reggae with $k=2$, you'll just get two of the rows that match reggae in the context window:

```
Based on this context:

 bob marley | i shot the sheriff | reggae
 ...

Answer the question:

Who are the reggae artists?

```


Instead, you probably want to create some tools that your agent can use as needed to answer more structured questions.

```python
def find_by_genre(genre):
  # Find artists by genre.
  matched = df[df.genre==genre]
  # ... return in some structured form that the LLM can use.

```

Then, you can use a ReAct agent to call these tools, e.g.

```python
agent = dspy.ReAct(
    signature="question -> answer",
    tools=[find_by_genre]
    max_iters=5  # Maximum number of tool calls (default is 10)
)
```
